In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count
from pyspark.sql.types import StructType, StructField, StringType



CRIME_SCHEMA = StructType(
    [
        StructField("code", StringType()),
        StructField("region", StringType()),
        StructField("category", StringType()),
    ]
)

input_df = spark.readStream.load(
    format="csv",
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/crime_data/input/",
    schema=CRIME_SCHEMA,
)

(
    input_df.writeStream.option(
        "checkpointLocation",
        "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/crime_data_input",
    )
    .trigger(availableNow=True)
    .outputMode("APPEND")
    .toTable("merit_catalog.quickstart_schema.crime_first_step")
)

input_df_02 = spark.readStream.table("merit_catalog.quickstart_schema.crime_first_step")

result_df_02 = input_df_02.groupBy("region").agg(count("*").alias("total_crime"))

(
    result_df_02.writeStream
    .outputMode("COMPLETE")  # Required for aggregation
    .option(
        "checkpointLocation",
        "/Volumes/merit_catalog/quickstart_schema/sandbox/checkpoints/crime_data_output"
    )
    .trigger(availableNow=True)
    .toTable("merit_catalog.quickstart_schema.crime_second_step")
)

